# Encoder–decoder (sequence to sequence)
The loss of the worked example by hand, then a small LSTM encoder–decoder trained on real English–French
sentence pairs with teacher forcing, greedy prediction word by word, teacher forcing against feeding the
model its own predictions.

In [1]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"          # hide TensorFlow's start-up messages
import re, time, random, collections
import numpy as np
import pandas as pd
import keras
import tensorflow as tf

np.set_printoptions(precision=4, suppress=True)
print(tf.config.list_physical_devices("GPU"))

I0000 00:00:1791075553.037111  251975 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## The loss of one training pair, by hand
The decoder's softmax gave the gold words "soch", "lo", "<end>" the probabilities 0.1, 0.1 and 0.4.
Categorical cross-entropy with a one-hot target keeps only -ln(probability of the gold word).

In [2]:
p_gold = np.array([0.1, 0.1, 0.4])
step_loss = -np.log(p_gold)
print("loss per step:", step_loss.round(3), " total:", step_loss.sum().round(3), " mean:", step_loss.mean().round(3))
# Keras gives the same mean (7 words: <start>, soch, lo, andar, aa, jao, <end>; the other probabilities are made up)
probs = np.array([[0.20, 0.10, 0.15, 0.30, 0.15, 0.08, 0.02],
                  [0.10, 0.05, 0.10, 0.20, 0.05, 0.40, 0.10],
                  [0.05, 0.05, 0.10, 0.10, 0.10, 0.20, 0.40]])
gold = np.array([1, 2, 6])                        # soch, lo, <end>
print("Keras mean loss:", float(keras.losses.SparseCategoricalCrossentropy()(gold, probs)))
print("base-10 log instead:", (-np.log10(p_gold)).round(3), (-np.log10(p_gold)).sum().round(3))

loss per step: [2.303 2.303 0.916]  total: 5.521  mean: 1.84


Keras mean loss: 1.840487003326416
base-10 log instead: [1.    1.    0.398] 2.398


## The data: English–French sentence pairs
The Keras example dataset (pairs from the Tatoeba project, via manythings.org). We keep sentences of up to
8 English words. Sentences are grouped by their English side, so a test sentence never appears in training
and all its French translations serve as references.

In [3]:
path = keras.utils.get_file("fra-eng.zip", origin="http://storage.googleapis.com/download.tensorflow.org/data/fra-eng.zip",
                            extract=True)
lines = open(os.path.join(path, "fra.txt"), encoding="utf8").read().strip().split("\n")
tok = lambda s: re.findall(r"[\w']+|[^\w\s]", s.lower())
groups = collections.defaultdict(list)
for line in lines:
    e, f = line.split("\t")[:2]
    e, f = tok(e), tok(f)
    if len(e) <= 8 and len(f) <= 10:
        groups[tuple(e)].append(f)
keys = sorted(groups)
random.Random(0).shuffle(keys)
test_keys, train_keys = keys[:1500], keys[1500:]
train = [(list(e), f) for e in train_keys for f in groups[e]]
random.Random(1).shuffle(train)
train = train[:40000]
print(len(lines), "pairs in the file;", len(keys), "distinct short English sentences;",
      len(train), "training pairs;", len(test_keys), "test sentences")
print(train[:3])

167130 pairs in the file; 82798 distinct short English sentences; 40000 training pairs; 1500 test sentences
[(['i', 'want', 'you', 'to', 'wear', 'this', 'one', '.'], ['je', 'veux', 'que', 'tu', 'portes', 'celle', '-', 'ci', '.']), (['will', 'you', 'lend', 'me', 'some', 'money', '?'], ['veux', '-', 'tu', 'me', 'prêter', 'de', "l'argent", '?']), (['the', 'house', 'is', 'empty', '.'], ['la', 'maison', 'est', 'vide', '.'])]


Vocabularies: the most frequent words of the training pairs, plus 4 special tokens.
Index 0 is padding, 1 stands for an unknown word.

In [4]:
def vocab(seqs, n):
    c = collections.Counter(w for s in seqs for w in s)
    words = ["<pad>", "<unk>", "<start>", "<end>"] + [w for w, _ in c.most_common(n)]
    return {w: i for i, w in enumerate(words)}, words

en_ix, en_w = vocab([e for e, _ in train], 6000)
fr_ix, fr_w = vocab([f for _, f in train], 8000)
LE, LF = 8, 12                                    # padded lengths: source, target incl. <start>/<end>

def encode(seqs, ix, L, add=False):
    X = np.zeros((len(seqs), L), "int32")
    for i, s in enumerate(seqs):
        s = ["<start>"] + s + ["<end>"] if add else s
        ids = [ix.get(w, 1) for w in s][:L]
        X[i, :len(ids)] = ids
    return X

Xe = encode([e for e, _ in train], en_ix, LE)
Yf = encode([f for _, f in train], fr_ix, LF, add=True)
Xd, Yd = Yf[:, :-1], Yf[:, 1:]                    # decoder input: <start> + gold; target: gold + <end>
print("vocabulary sizes:", len(en_w), len(fr_w))
print("one pair:", train[0])
print("encoder input :", Xe[0])
print("decoder input :", Xd[0], [fr_w[i] for i in Xd[0] if i])
print("decoder target:", Yd[0], [fr_w[i] for i in Yd[0] if i])

vocabulary sizes: 6004 8004
one pair: (['i', 'want', 'you', 'to', 'wear', 'this', 'one', '.'], ['je', 'veux', 'que', 'tu', 'portes', 'celle', '-', 'ci', '.'])
encoder input : [  5  38   6   8 810  15  71   4]
decoder input : [  2   5  49  14  12 849 743   6 383   4   3] ['<start>', 'je', 'veux', 'que', 'tu', 'portes', 'celle', '-', 'ci', '.', '<end>']
decoder target: [  5  49  14  12 849 743   6 383   4   3   0] ['je', 'veux', 'que', 'tu', 'portes', 'celle', '-', 'ci', '.', '<end>']


## The model: embedding + LSTM encoder, embedding + LSTM decoder, softmax over the French words
The encoder's final hidden and cell states (the context vector) become the decoder's initial states.

In [5]:
D, H = 128, 256

def build(seed):
    keras.utils.set_random_seed(seed)
    ei = keras.Input((None,), dtype="int32", name="english")
    di = keras.Input((None,), dtype="int32", name="french_in")
    x = keras.layers.Embedding(len(en_w), D, mask_zero=True, name="emb_en")(ei)
    _, h, c = keras.layers.LSTM(H, return_state=True, name="encoder")(x)
    y = keras.layers.Embedding(len(fr_w), D, mask_zero=True, name="emb_fr")(di)
    y = keras.layers.LSTM(H, return_sequences=True, name="decoder")(y, initial_state=[h, c])
    out = keras.layers.Dense(len(fr_w), activation="softmax", name="softmax")(y)
    m = keras.Model([ei, di], out)
    m.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return m

model = build(0)
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ english             │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ french_in           │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ emb_en (Embedding)  │ (None, None, 128) │    768,512 │ english[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, None)      │          0 │ english[0][0]     │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ emb_fr (Embedding)  │ (None, None, 128) │  1,024,512 │ french_in[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder (LSTM)      │ [(None, 256),     │    394,240 │ emb_en[0][0],     │
│                     │ (None, 256),      │            │ not_equal[0][0]   │
│                     │ (None, 256)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder (LSTM)      │ (None, None, 256) │    394,240 │ emb_fr[0][0],     │
│                     │                   │            │ encoder[0][1],    │
│                     │                   │            │ encoder[0][2]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ softmax (Dense)     │ (None, None,      │  2,057,028 │ decoder[0][0]     │
│                     │ 8004)             │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 4,638,532 (17.69 MB)

 Trainable params: 4,638,532 (17.69 MB)

 Non-trainable params: 0 (0.00 B)

## BLEU, to score the translations
Corpus BLEU-4 (Papineni et al. 2002): clipped n-gram precisions for n = 1..4, their geometric mean,
times a brevity penalty; several references per sentence allowed. On a 0–100 scale.

In [6]:
def ngrams(s, n):
    return collections.Counter(tuple(s[i:i + n]) for i in range(len(s) - n + 1))

def bleu(hyps, refss):
    match, total = np.zeros(4), np.zeros(4)
    hyp_len = ref_len = 0
    for h, refs in zip(hyps, refss):
        hyp_len += len(h)
        ref_len += min((abs(len(r) - len(h)), len(r)) for r in refs)[1]
        for n in range(1, 5):
            hc = ngrams(h, n)
            best = collections.Counter()
            for r in refs:
                best |= ngrams(r, n)
            match[n - 1] += sum(min(c, best[g]) for g, c in hc.items())
            total[n - 1] += max(len(h) - n + 1, 0)
    if match.min() == 0:
        return 0.0
    bp = 1.0 if hyp_len > ref_len else np.exp(1 - ref_len / max(hyp_len, 1))
    return 100 * bp * np.exp(np.mean(np.log(match / total)))

assert abs(bleu([["a", "b", "c", "d"]], [[["a", "b", "c", "d"]]]) - 100) < 1e-9

## Training with teacher forcing
The decoder input is the gold French sentence shifted right by one (`Xd`), so every step receives the
correct previous word whatever the model predicted. 15 epochs, Adam, batch size 128.

In [7]:
t0 = time.time()
hist = model.fit([Xe, Xd], Yd, batch_size=128, epochs=15, validation_split=0.05, verbose=2).history
print("training time (s):", round(time.time() - t0))

Epoch 1/15


297/297 - 25s - 83ms/step - accuracy: 0.3211 - loss: 4.6027 - val_accuracy: 0.3999 - val_loss: 3.8400


Epoch 2/15


297/297 - 18s - 61ms/step - accuracy: 0.4150 - loss: 3.6558 - val_accuracy: 0.4282 - val_loss: 3.5119


Epoch 3/15


...

Epoch 14/15


297/297 - 25s - 83ms/step - accuracy: 0.6764 - loss: 1.4666 - val_accuracy: 0.6232 - val_loss: 2.0213


Epoch 15/15


297/297 - 26s - 86ms/step - accuracy: 0.6901 - loss: 1.3783 - val_accuracy: 0.6271 - val_loss: 1.9963


training time (s): 326


## Prediction: greedy decoding, word by word
Two small models share the trained layers: the encoder gives the context vector; the decoder step
takes one word and the states and returns probabilities and new states. The predicted word is fed back.

In [8]:
def inference_models(m):
    """Encoder model and one-step decoder model that reuse the trained layers of m."""
    enc = keras.Model(m.inputs[0], m.get_layer("encoder").output[1:])
    d_in, h_in, c_in = keras.Input((None,), dtype="int32"), keras.Input((H,)), keras.Input((H,))
    y = m.get_layer("emb_fr")(d_in)
    y, h, c = keras.layers.LSTM(H, return_sequences=True, return_state=True)(y, initial_state=[h_in, c_in])
    lstm = [l for l in [*keras.Model([d_in, h_in, c_in], y).layers] if isinstance(l, keras.layers.LSTM)][0]
    lstm.set_weights(m.get_layer("decoder").get_weights())
    step = keras.Model([d_in, h_in, c_in], [m.get_layer("softmax")(y), h, c])
    return enc, step

def translate(models, sentences, keep_steps=False):
    """Greedy decoding for a batch of tokenised English sentences."""
    enc_model, step_model = models
    h, c = enc_model.predict(encode(sentences, en_ix, LE), verbose=0)
    word = np.full((len(sentences), 1), fr_ix["<start>"], "int32")
    out, done, steps = [[] for _ in sentences], np.zeros(len(sentences), bool), []
    for t in range(LF):
        p, h, c = step_model.predict([word, h, c], verbose=0)
        p = p[:, 0]
        word = p.argmax(-1).reshape(-1, 1).astype("int32")
        if keep_steps:
            steps.append(p[0])
        for i, w in enumerate(word[:, 0]):
            if not done[i]:
                if fr_w[w] == "<end>":
                    done[i] = True
                else:
                    out[i].append(fr_w[w])
        if done.all():
            break
    return (out, steps) if keep_steps else out

test_en = [list(k) for k in test_keys]
test_refs = [groups[k] for k in test_keys]
models = inference_models(model)
hyp = translate(models, test_en)
print("test BLEU (teacher forcing, 15 epochs):", round(bleu(hyp, test_refs), 1))
for e, h, r in list(zip(test_en, hyp, test_refs))[:12]:
    print(" ".join(e), "->", " ".join(h), "   | reference:", " ".join(r[0]))

test BLEU (teacher forcing, 15 epochs): 13.1
that doesn't help me . -> ça ne me dérange pas .    | reference: ça ne m'aide pas .
i like your house . -> j'aime votre maison .    | reference: j'aime votre maison .
that guy is completely nuts ! -> ce mec est complètement - moi !    | reference: ce mec est complètement zinzin !
i'm sorry , but i don't understand . -> je suis désolé , je ne suis pas content .    | reference: je suis désolé mais je ne comprends pas .
she accompanied her friend to the concert . -> elle a <unk> son livre .    | reference: elle accompagna son ami au concert .
we have an unusual relationship . -> nous avons un accord à la nuit .    | reference: nous avons une relation inhabituelle .
what's his most recent novel ? -> quel est la meilleure chambre de la maison ?    | reference: quel est son roman le plus récent ?
the pain has mostly gone away . -> la douleur a été fabriquée en prison .    | reference: la douleur a en majeure partie disparu .
they have everything t

One test sentence step by step: the five most likely words at every step, and the word fed back.

In [9]:
demo = "i think you're right .".split()
out, steps = translate(models, [demo], keep_steps=True)
rows = []
for t, p in enumerate(steps[:len(out[0]) + 1]):
    top = p.argsort()[::-1][:5]
    for rank, k in enumerate(top):
        rows.append(dict(step=t + 1, rank=rank + 1, word=fr_w[k], prob=round(float(p[k]), 4)))
    print(t + 1, [(fr_w[k], round(float(p[k]), 3)) for k in top])
print(" ".join(demo), "->", " ".join(out[0]))
pd.DataFrame(rows).to_csv("data/decode_steps.csv", index=False)
pd.DataFrame(dict(english=[" ".join(e) for e in test_en[:30]], predicted=[" ".join(h) for h in hyp[:30]],
                  reference=[" ".join(r[0]) for r in test_refs[:30]])).to_csv("data/translations.csv", index=False)

1 [('je', 0.993), ("j'ai", 0.001), ("j'y", 0.001), ("j'ignorais", 0.0), ("j'imagine", 0.0)]
2 [('pense', 0.887), ('crois', 0.041), ('vous', 0.021), ('suppose', 0.011), ('te', 0.007)]
3 [('que', 0.937), ('vraiment', 0.007), ('juste', 0.006), ('ce', 0.004), ("qu'il", 0.004)]
4 [('vous', 0.549), ('tu', 0.399), ('ce', 0.015), ("c'est", 0.009), ('ça', 0.008)]
5 [('êtes', 0.754), ('avez', 0.067), ('soyez', 0.014), ("l'avez", 0.012), ('vous', 0.01)]
6 [('sérieux', 0.059), ('raison', 0.056), ('jaloux', 0.036), ('là', 0.036), ('bien', 0.029)]
7 [('.', 0.995), ('!', 0.004), (',', 0.0), ('?', 0.0), ('sérieuses', 0.0)]
8 [('<end>', 1.0), ('.', 0.0), ('»', 0.0), ('ça', 0.0), ('<unk>', 0.0)]
i think you're right . -> je pense que vous êtes sérieux .


## Teacher forcing against feeding back the model's own predictions
The same architecture written as an explicit loop over the decoder steps. With `teacher=True` the next
input is the gold word; with `teacher=False` it is the model's own most likely word (no gradient flows
through that choice). Both are trained for 10 epochs on the same data, 3 seeds each, and scored after
every epoch with greedy decoding on the test sentences (the same loop with `teacher=False`).

In [10]:
loss_fn = keras.losses.SparseCategoricalCrossentropy(from_logits=True, reduction="none")
Xe_test = encode(test_en, en_ix, LE)

def make_layers(seed):
    keras.utils.set_random_seed(seed)
    return dict(emb_e=keras.layers.Embedding(len(en_w), D, mask_zero=True), enc=keras.layers.LSTM(H, return_state=True),
                emb_d=keras.layers.Embedding(len(fr_w), D), cell=keras.layers.LSTMCell(H), out=keras.layers.Dense(len(fr_w)))

def unroll(L, xe, first, gold, teacher, steps):
    _, h, c = L["enc"](L["emb_e"](xe))
    word, logits = first, []
    for t in range(steps):
        o, (h, c) = L["cell"](L["emb_d"](word), [h, c])
        lg = L["out"](o)
        logits.append(lg)
        if teacher and t + 1 < steps:
            word = gold[:, t + 1]
        else:
            word = tf.stop_gradient(tf.argmax(lg, -1, output_type=tf.int32))
    return tf.stack(logits, 1)

def greedy_words(L):
    first = tf.fill([len(Xe_test)], fr_ix["<start>"])
    ids = tf.argmax(unroll(L, Xe_test, first, None, False, LF - 1), -1).numpy()
    out = []
    for row in ids:
        s = []
        for k in row:
            if fr_w[k] == "<end>":
                break
            s.append(fr_w[k])
        out.append(s)
    return out

def train_loop(teacher, seed, epochs=10):
    L = make_layers(seed)
    opt = keras.optimizers.Adam()
    unroll(L, Xe[:2], Xd[:2, 0], Xd[:2], True, LF - 1)
    params = [v for layer in L.values() for v in layer.trainable_variables]

    @tf.function
    def step(xe, xd, yd):
        with tf.GradientTape() as tape:
            lg = unroll(L, xe, xd[:, 0], xd, teacher, LF - 1)
            mask = tf.cast(yd > 0, tf.float32)
            loss = tf.reduce_sum(loss_fn(yd, lg) * mask) / tf.reduce_sum(mask)
        opt.apply_gradients(zip(tape.gradient(loss, params), params))
        return loss

    ds = tf.data.Dataset.from_tensor_slices((Xe, Xd, Yd)).shuffle(len(Xe), seed=seed).batch(128)
    curve = []
    for ep in range(epochs):
        losses = [float(step(a, b, c)) for a, b, c in ds]
        curve.append(dict(epoch=ep + 1, teacher=teacher, seed=seed, train_loss=np.mean(losses),
                          test_bleu=bleu(greedy_words(L), test_refs)))
    print(f"teacher={teacher} seed={seed}: BLEU after epochs 1..{epochs}:", [round(float(r['test_bleu']), 1) for r in curve])
    return curve

t0 = time.time()
curves = pd.DataFrame([r for teacher in (True, False) for seed in range(3) for r in train_loop(teacher, seed)])
print("time (s):", round(time.time() - t0))
curves.to_csv("data/teacher_forcing.csv", index=False)
print(curves.groupby(["teacher", "epoch"]).test_bleu.mean().unstack(0).round(1))

teacher=True seed=0: BLEU after epochs 1..10: [0.9, 1.2, 2.6, 3.3, 4.7, 6.3, 8.0, 8.4, 10.1, 10.3]


teacher=True seed=1: BLEU after epochs 1..10: [1.0, 1.0, 2.4, 3.3, 5.0, 6.0, 7.2, 8.4, 9.7, 9.9]


teacher=True seed=2: BLEU after epochs 1..10: [1.0, 1.4, 2.3, 3.2, 4.8, 6.4, 7.5, 8.4, 9.7, 10.3]


teacher=False seed=0: BLEU after epochs 1..10: [0.0, 0.0, 0.0, 2.7, 4.6, 6.3, 7.3, 8.6, 9.3, 10.4]


teacher=False seed=1: BLEU after epochs 1..10: [0.0, 0.0, 0.0, 2.7, 4.8, 6.0, 7.4, 8.1, 9.1, 10.2]


teacher=False seed=2: BLEU after epochs 1..10: [0.0, 0.0, 0.0, 2.0, 4.9, 5.7, 6.5, 8.1, 8.1, 9.0]
time (s): 1348
teacher  False  True 
epoch                
1          0.0    1.0
2          0.0    1.2
3          0.0    2.5
4          2.5    3.3
5          4.8    4.8
6          6.0    6.2
7          7.1    7.6
8          8.3    8.4
9          8.9    9.9
10         9.9   10.2
